# 5.3 结构化输出与约束解码 / Structured output and constrained decoding

用状态机在每一步屏蔽不合法的 token，使输出保证符合给定的格式。

Mask illegal tokens at every step with a state machine, so that the output is guaranteed to follow a given format.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 用状态机屏蔽 logits / Mask logits with a state machine

实现在字符级状态机上行走、计算某个状态下允许的 token，以及带约束的贪心解码。测试先在一个很小的词表上检查，再在 GPT-2 上用两个手写的状态机（只能回答 yes 或 no，只能输出 1 到 3 位数字）运行。运行后请对比打印出的无约束输出和有约束输出，并注意有约束的答案是否正确。

Implement walking on a character-level state machine, computing the tokens allowed in a state, and greedy decoding under constraints. The tests first check on a tiny vocabulary and then run on GPT-2 with two hand-written state machines (answering only yes or no, producing only 1 to 3 digits). Compare the printed unconstrained and constrained outputs and note whether the constrained answers are correct.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# A deterministic finite automaton (DFA) over characters:
#   dfa = {"trans": {state: {char: next_state}}, "accept": set_of_states}      the start state is 0
# It describes which strings are allowed. The model produces tokens, and one token is usually several
# characters, so a token is allowed in a state if ALL of its characters can be followed in the DFA.


def dfa_walk(dfa, state, text):
    """Follow the characters of `text` from `state`. Returns the state reached, or None if some
    character has no transition (the text is not allowed here)."""
    # TODO 1
    raise NotImplementedError


def allowed_tokens(dfa, state, token_strings):
    """Ids of all tokens that may come next in `state`.

    token_strings[i] is the text of token i. A token is allowed if its text is not empty
    and dfa_walk does not return None.
    """
    # TODO 2
    raise NotImplementedError


@torch.no_grad()
def constrained_greedy(model, token_strings, dfa, input_ids, max_new_tokens, eos_id):
    """Greedy decoding in which only tokens allowed by the DFA can be chosen. Returns the generated text.

    At every step:
      1. allowed = allowed_tokens(dfa, state, token_strings); cache this per state, it is slow.
         If the state is accepting, the end-of-sequence token eos_id is allowed as well.
      2. Set the logits of all other tokens to -inf and take the argmax.
      3. If eos_id was chosen, stop. Otherwise append the token text and advance the state with dfa_walk.
      4. If the new state is accepting and has no outgoing transitions, stop: the output is complete.
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Two small automata written by hand.
YES_NO = {"trans": {0: {"y": 1, "n": 4}, 1: {"e": 2}, 2: {"s": 3}, 4: {"o": 3}}, "accept": {3}}          # yes | no
DIGITS = "0123456789"
NUMBER = {"trans": {0: {d: 1 for d in DIGITS}, 1: {d: 2 for d in DIGITS}, 2: {d: 3 for d in DIGITS}}, "accept": {1, 2, 3}}   # 1 to 3 digits

# 1. Walking
assert dfa_walk(YES_NO, 0, "ye") == 2 and dfa_walk(YES_NO, 0, "yes") == 3 and dfa_walk(YES_NO, 2, "s") == 3
assert dfa_walk(YES_NO, 0, "yea") is None and dfa_walk(YES_NO, 3, "s") is None and dfa_walk(YES_NO, 0, "") == 0

# 2. Allowed tokens on a tiny vocabulary: whole words, pieces and characters all work.
vocab = ["yes", "y", "es", "e", "s", "no", "n", "o", "maybe", " yes", "", "ye", "yess"]
names = lambda ids: sorted(vocab[i] for i in ids)
assert names(allowed_tokens(YES_NO, 0, vocab)) == ["n", "no", "y", "ye", "yes"]
assert names(allowed_tokens(YES_NO, 1, vocab)) == ["e", "es"]
assert names(allowed_tokens(YES_NO, 3, vocab)) == []

# 3. On GPT-2
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
strings = tok.batch_decode([[i] for i in range(len(tok))])
n_first = len(allowed_tokens(NUMBER, 0, strings))
assert 500 < n_first < 1110, n_first            # the tokens made of 1 to 3 digits, out of 50257

rows = []
for prompt in ["Question: Is Paris the capital of France? Answer (yes or no):", "Question: Is the Moon made of cheese? Answer (yes or no):"]:
    ids = tok(prompt, return_tensors="pt").input_ids
    free = tok.decode(model.generate(ids, do_sample=False, max_new_tokens=6, pad_token_id=tok.eos_token_id)[0, ids.shape[1]:])
    out = constrained_greedy(model, strings, YES_NO, ids, 6, tok.eos_token_id)
    assert out in ("yes", "no"), out
    rows.append((prompt, free, out))
for prompt in ["The number of days in a week is", "The year the Second World War ended was"]:
    ids = tok(prompt, return_tensors="pt").input_ids
    free = tok.decode(model.generate(ids, do_sample=False, max_new_tokens=6, pad_token_id=tok.eos_token_id)[0, ids.shape[1]:])
    out = constrained_greedy(model, strings, NUMBER, ids, 6, tok.eos_token_id)
    assert re.fullmatch(r"[0-9]{1,3}", out), out
    rows.append((prompt, free, out))

print(f"m53_fsm: all tests passed ({n_first} of {len(strings)} tokens are allowed at the start of a number)")
for prompt, free, out in rows:
    print(f"  {prompt!r}")
    print(f"      unconstrained: {free!r}")
    print(f"      constrained:   {out!r}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`dfa['trans'].get(state, {}).get(ch)` 在没有对应的转移时返回 `None`。

`dfa['trans'].get(state, {}).get(ch)` returns `None` when there is no matching transition.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

屏蔽可以写成先建一个全为负无穷的向量，把允许的位置设为 0，再与 logits 相加。

The mask can be written as a vector of minus infinity with zeros at the allowed positions, added to the logits.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

每个状态下允许的 token 只需计算一次，用一个字典按状态保存。

The allowed tokens of each state need computing only once and can be kept in a dictionary keyed by state.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# A deterministic finite automaton (DFA) over characters:
#   dfa = {"trans": {state: {char: next_state}}, "accept": set_of_states}      the start state is 0


def dfa_walk(dfa, state, text):
    for ch in text:
        state = dfa["trans"].get(state, {}).get(ch)
        if state is None:
            return None
    return state


def allowed_tokens(dfa, state, token_strings):
    return [i for i, s in enumerate(token_strings) if s and dfa_walk(dfa, state, s) is not None]


@torch.no_grad()
def constrained_greedy(model, token_strings, dfa, input_ids, max_new_tokens, eos_id):
    state, text, cache = 0, "", {}
    for _ in range(max_new_tokens):
        if state not in cache:
            cache[state] = allowed_tokens(dfa, state, token_strings)
        allowed = list(cache[state])
        if state in dfa["accept"]:
            allowed.append(eos_id)                              # a complete output may stop here
        if not allowed:
            break
        logits = model(input_ids).logits[0, -1]
        mask = torch.full_like(logits, float("-inf"))
        mask[allowed] = 0.0
        next_id = (logits + mask).argmax().item()
        if next_id == eos_id and state in dfa["accept"]:
            break
        text += token_strings[next_id]
        state = dfa_walk(dfa, state, token_strings[next_id])
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]])], dim=1)
        if state in dfa["accept"] and not dfa["trans"].get(state):
            break                                               # nothing can follow, so the output is finished
    return text
```

</details>

## 2. 把 JSON Schema 的子集编译成状态机 / Compile a subset of JSON Schema into a state machine

实现四种构件（固定文本、多选一、整数、有长度上限的字符串），再用它们把一个只含简单字段的对象模式编译成字符级状态机。测试会检查每种构件接受和拒绝的字符串，并在状态机上随机行走 300 次，确认它接受的每个字符串都能被 `json.loads` 解析并且符合模式。

Implement four building blocks (fixed text, one of several options, an integer, a string with a maximum length) and use them to compile an object schema with simple fields into a character-level state machine. The tests check which strings each block accepts and rejects and perform 300 random walks on the state machine, confirming that every string it accepts is parsed by `json.loads` and conforms to the schema.

In [ ]:
import json

STRING_CHARS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?-'"
DIGITS = "0123456789"


class Builder:
    """Builds a DFA step by step. Nothing to change in this class."""

    def __init__(self):
        self.trans = {0: {}}

    def new_state(self):
        s = len(self.trans)
        self.trans[s] = {}
        return s

    def step(self, states, chars):
        """From every state in `states`, each character in `chars` leads to one new shared state.
        Returns that state as a one-element set."""
        target = self.new_state()
        for s in states:
            for ch in chars:
                assert ch not in self.trans[s], f"state {s} already has a transition on {ch!r}"
                self.trans[s][ch] = target
        return {target}


# Every function below takes the Builder and the set of states reached so far, adds transitions,
# and returns the set of states in which its piece of text can end.


def literal(b, states, text):
    """A fixed piece of text, one b.step per character."""
    # TODO 1
    raise NotImplementedError


def choice(b, states, options):
    """One of several fixed strings, for example ["true", "false"].

    Options may share a prefix ("red", "rose"), so build a trie: for each option walk from the start
    state, follow a transition if the character already has one, otherwise create a new state with
    b.new_state() and add the transition to b.trans directly. Returns the set of end states.
    `states` always contains exactly one state here.
    """
    # TODO 2
    raise NotImplementedError


def integer(b, states, max_digits=4):
    """A non-negative integer without leading zeros: "0", or a digit 1-9 followed by up to
    max_digits - 1 further digits. Returns all states in which the number may end."""
    # TODO 3
    raise NotImplementedError


def free_string(b, states, max_length):
    """A JSON string of at most max_length characters from STRING_CHARS, including both quotes.

    The closing quote may come after 0, 1, ..., max_length characters and leads to one shared state.
    """
    # TODO 4
    raise NotImplementedError


def compile_schema(schema):
    """Compile a small subset of JSON Schema into a DFA {"trans": ..., "accept": ...}.

    Supported: an object whose properties are all required and appear in the given order, with values
      {"enum": [strings]}, {"type": "boolean"}, {"type": "integer", "maxDigits": n}, {"type": "string", "maxLength": n}.
    The output format is fixed:  {"name": value, "name2": value}
    (one space after each colon and comma; json.dumps(name) gives the quoted key).
    """
    # TODO 5
    raise NotImplementedError

In [ ]:
# 检查 / checks
import json
import random


def walk(dfa, text):
    s = 0
    for ch in text:
        s = dfa["trans"].get(s, {}).get(ch)
        if s is None:
            return None
    return s

accepts = lambda dfa, text: walk(dfa, text) in dfa["accept"]

def piece(build):
    b = Builder()
    ends = build(b, {0})
    return {"trans": b.trans, "accept": set(ends)}

# 1. The building blocks
d = piece(lambda b, s: literal(b, s, "abc"))
assert accepts(d, "abc") and not accepts(d, "ab") and not accepts(d, "abcd")
d = piece(lambda b, s: choice(b, s, ["true", "false"]))
assert accepts(d, "true") and accepts(d, "false") and not accepts(d, "tru") and not accepts(d, "truefalse")
d = piece(lambda b, s: choice(b, s, ['"red"', '"rose"', '"blue"']))          # shared prefix
assert all(accepts(d, x) for x in ['"red"', '"rose"', '"blue"']) and not accepts(d, '"ro"') and not accepts(d, '"rede"')
d = piece(lambda b, s: integer(b, s, 3))
assert all(accepts(d, x) for x in ["0", "7", "42", "999"])
assert not any(accepts(d, x) for x in ["", "007", "01", "1000", "-1", "4.2"])
d = piece(lambda b, s: free_string(b, s, 5))
assert all(accepts(d, x) for x in ['""', '"a"', '"Hi yo"'])
assert not any(accepts(d, x) for x in ['"', '"abcdef"', '"a"b"', 'abc', '"a\\"'])

# 2. A whole schema
SCHEMA = {"type": "object", "properties": {
    "sentiment": {"enum": ["positive", "negative", "neutral"]},
    "stars": {"type": "integer", "maxDigits": 1},
    "recommend": {"type": "boolean"},
    "summary": {"type": "string", "maxLength": 12}}}
dfa = compile_schema(SCHEMA)
good = '{"sentiment": "negative", "stars": 2, "recommend": false, "summary": "Too slow."}'
assert accepts(dfa, good) and json.loads(good)["stars"] == 2
for bad in [good[:-1], good + "}", good.replace("negative", "bad"), good.replace("false", "False"), good.replace(": 2", ': "2"'),
            good.replace('"stars": 2, ', ""), good.replace("Too slow.", "This summary is far too long"), good.replace('", "stars', '","stars')]:
    assert not accepts(dfa, bad), bad

# 3. Every string the DFA accepts is valid JSON that satisfies the schema. Check by random walks.
def conforms(obj):
    return (isinstance(obj, dict) and list(obj) == ["sentiment", "stars", "recommend", "summary"]
            and obj["sentiment"] in ("positive", "negative", "neutral")
            and type(obj["stars"]) is int and 0 <= obj["stars"] <= 9
            and type(obj["recommend"]) is bool and type(obj["summary"]) is str and len(obj["summary"]) <= 12)

rng = random.Random(0)
seen = set()
for _ in range(300):
    s, text = 0, ""
    while dfa["trans"].get(s):
        if s in dfa["accept"] and rng.random() < 0.5:
            break
        ch = rng.choice(sorted(dfa["trans"][s]))
        text, s = text + ch, dfa["trans"][s][ch]
    assert s in dfa["accept"], "every path must be able to finish"
    assert conforms(json.loads(text)), text
    seen.add(text)
assert len(seen) > 250

# 4. The automaton is finite and has no loops, so generation is guaranteed to end.
longest = {}
def depth(s):
    if s not in longest:
        longest[s] = -1                                        # a loop would hit this marker
        nxt = [depth(t) for t in dfa["trans"].get(s, {}).values()]
        assert -1 not in nxt, "the DFA must not contain a loop"
        longest[s] = 1 + max(nxt) if nxt else 0
    return longest[s]
print(f"m53_schema: all tests passed ({len(dfa['trans'])} states, longest output {depth(0)} characters)")
print("  one random string it accepts:", sorted(seen)[0])

<details><summary><b>提示 1 / Hint 1</b></summary>

`b.step(states, chars)` 让 `states` 中的每个状态都经由 `chars` 中的任一字符到达同一个新状态，并返回只含这个新状态的集合。

`b.step(states, chars)` leads every state in `states` through any character in `chars` to one new shared state and returns the set containing just that state.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

整数可以在读完 1 位、2 位直到最大位数之后结束，所以要把这些状态都放进返回的集合里。字符串同理，结束引号可以从任意长度的状态出发。

An integer may end after 1 digit, 2 digits and so on up to the maximum, so all of these states go into the returned set. Likewise for the string, whose closing quote may start from the state of any length.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`compile_schema` 中每个字段先接一段固定文本 `(", " if i else "") + json.dumps(name) + ": "`，再接对应类型的构件。

In `compile_schema` every field first appends the fixed text `(", " if i else "") + json.dumps(name) + ": "` and then the block for its type.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import json

STRING_CHARS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?-'"
DIGITS = "0123456789"


class Builder:
    """Builds a DFA step by step. Nothing to change in this class."""

    def __init__(self):
        self.trans = {0: {}}

    def new_state(self):
        s = len(self.trans)
        self.trans[s] = {}
        return s

    def step(self, states, chars):
        """From every state in `states`, each character in `chars` leads to one new shared state.
        Returns that state as a one-element set."""
        target = self.new_state()
        for s in states:
            for ch in chars:
                assert ch not in self.trans[s], f"state {s} already has a transition on {ch!r}"
                self.trans[s][ch] = target
        return {target}


def literal(b, states, text):
    for ch in text:
        states = b.step(states, ch)
    return states


def choice(b, states, options):
    (start,) = states                                      # a choice always starts from a single state
    ends = set()
    for option in options:
        s = start
        for ch in option:                                  # walk the shared prefix, branch where it ends
            if ch not in b.trans[s]:
                b.trans[s][ch] = b.new_state()
            s = b.trans[s][ch]
        ends.add(s)
    return ends


def integer(b, states, max_digits=4):
    zero = b.step(states, "0")                             # "0" alone, no leading zeros otherwise
    cur = b.step(states, "123456789")
    ends = zero | cur
    for _ in range(max_digits - 1):
        cur = b.step(cur, DIGITS)
        ends |= cur
    return ends


def free_string(b, states, max_length):
    cur = b.step(states, '"')
    inside = set(cur)                                      # after 0 characters
    for _ in range(max_length):
        cur = b.step(cur, STRING_CHARS)
        inside |= cur
    return b.step(inside, '"')                             # the closing quote is possible after any length


def compile_schema(schema):
    b = Builder()
    states = literal(b, {0}, "{")
    for i, (name, spec) in enumerate(schema["properties"].items()):
        states = literal(b, states, (", " if i else "") + json.dumps(name) + ": ")
        if "enum" in spec:
            states = choice(b, states, [json.dumps(v) for v in spec["enum"]])
        elif spec["type"] == "boolean":
            states = choice(b, states, ["true", "false"])
        elif spec["type"] == "integer":
            states = integer(b, states, spec.get("maxDigits", 4))
        elif spec["type"] == "string":
            states = free_string(b, states, spec.get("maxLength", 20))
        else:
            raise ValueError(f"unsupported: {spec}")
    states = literal(b, states, "}")
    return {"trans": b.trans, "accept": set(states)}
```

</details>

## 3. 对比约束前后的格式错误率 / Compare format error rates with and without constraints

实现从文本中提取 JSON、检查它是否符合模式、统计格式错误率和某个字段的正确率。测试随后在 GPT-2 上对 8 条评论运行四种设置，即有无示例与有无约束的组合。运行大约需要一分钟。请查看打印出的表格，区分约束改变了哪一列、没有改变哪一列。

Implement extracting JSON from text, checking whether it conforms to the schema, and computing the format error rate and the accuracy of one field. The tests then run four settings on GPT-2 for 8 reviews, the combinations of with and without examples and with and without constraints. The run takes about a minute. Look at the printed table and distinguish which column the constraint changes and which it does not.

需要联网下载模型。/ Downloads a model.

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# dfa_walk, allowed_tokens, constrained_greedy (exercise 1) and compile_schema (exercise 2) are defined above.


def extract_json(text):
    """The first {...} in `text`, parsed. Returns the object, or None if there is none or it does not parse.

    Take the text from the first "{" up to and including the first "}" (the schemas here are not nested)
    and pass it to json.loads; catch json.JSONDecodeError.
    """
    # TODO 1
    raise NotImplementedError


def conforms(obj, schema):
    """True if obj is a dict with exactly the properties of the schema and every value has the right type.

    The value kinds are those of exercise 2: {"enum": [...]}, boolean, integer, string.
    Careful: in Python, isinstance(True, int) is True. Compare with type(value) is int.
    """
    # TODO 2
    raise NotImplementedError


def format_error_rate(outputs, schema):
    """Fraction of output texts that contain no JSON object conforming to the schema."""
    # TODO 3
    raise NotImplementedError


def field_accuracy(outputs, field, labels):
    """Fraction of outputs whose parsed object has obj[field] == label. Unparseable outputs count as wrong."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SCHEMA = {"type": "object", "properties": {
    "sentiment": {"enum": ["positive", "negative"]},
    "stars": {"type": "integer", "maxDigits": 1},
    "recommend": {"type": "boolean"}}}

# 1. The checking functions
assert extract_json('Sure! {"a": 1} and more {"b": 2}') == {"a": 1}
assert extract_json("no json here") is None and extract_json('{"a": 1') is None and extract_json("{'a': 1}") is None
ok = {"sentiment": "positive", "stars": 4, "recommend": True}
assert conforms(ok, SCHEMA)
for bad in [None, [], {**ok, "sentiment": "great"}, {**ok, "stars": "4"}, {**ok, "stars": True}, {**ok, "recommend": "yes"},
            {**ok, "extra": 1}, {"sentiment": "positive", "stars": 4}]:
    assert not conforms(bad, SCHEMA), bad
assert format_error_rate([json.dumps(ok), "oops", '{"stars": 4}', "x " + json.dumps(ok) + " y"], SCHEMA) == 0.5
assert field_accuracy([json.dumps(ok), "oops", json.dumps(ok)], "sentiment", ["positive", "positive", "negative"]) == 1 / 3

# 2. The experiment on GPT-2
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
strings = tok.batch_decode([[i] for i in range(len(tok))])
dfa = compile_schema(SCHEMA)

REVIEWS = [("The food was wonderful and the staff were friendly.", "positive"), ("Terrible service, I will never come back.", "negative"),
           ("I loved every minute of this film.", "positive"), ("The battery died after two days. Very disappointing.", "negative"),
           ("An excellent book, beautifully written.", "positive"), ("The room was dirty and the bed was broken.", "negative"),
           ("Great value and fast delivery, highly recommended.", "positive"), ("Boring, slow and far too long.", "negative")]
labels = [label for _, label in REVIEWS]
ZERO = 'Review: {r}\nJSON with the fields sentiment, stars and recommend:'
SHOT = ('Review: The pizza was cold and the waiter was rude.\nJSON: {{"sentiment": "negative", "stars": 1, "recommend": false}}\n\n'
        'Review: A lovely hotel with a fantastic view.\nJSON: {{"sentiment": "positive", "stars": 5, "recommend": true}}\n\n'
        'Review: {r}\nJSON:')

def free(prompt):
    ids = tok(prompt, return_tensors="pt").input_ids
    return tok.decode(model.generate(ids, do_sample=False, max_new_tokens=24, pad_token_id=tok.eos_token_id)[0, ids.shape[1]:])

def constrained(prompt):
    ids = tok(prompt + " ", return_tensors="pt").input_ids
    return constrained_greedy(model, strings, dfa, ids, 40, tok.eos_token_id)

runs = {"zero-shot, unconstrained": [free(ZERO.format(r=r)) for r, _ in REVIEWS],
        "two-shot, unconstrained": [free(SHOT.format(r=r)) for r, _ in REVIEWS],
        "zero-shot, constrained": [constrained(ZERO.format(r=r)) for r, _ in REVIEWS],
        "two-shot, constrained": [constrained(SHOT.format(r=r)) for r, _ in REVIEWS]}
err = {k: format_error_rate(v, SCHEMA) for k, v in runs.items()}
acc = {k: field_accuracy(v, "sentiment", labels) for k, v in runs.items()}

assert err["zero-shot, constrained"] == 0 and err["two-shot, constrained"] == 0, "constrained decoding cannot produce a format error"
assert err["zero-shot, unconstrained"] > 0.5, "without examples or constraints GPT-2 rarely produces the format"
assert all(dfa_walk(dfa, 0, t) in dfa["accept"] for t in runs["zero-shot, constrained"])

print("m53_compare: all tests passed")
print("  setting                      format errors   sentiment correct")
for k in runs:
    print(f"  {k:27s}  {err[k]:8.0%}        {acc[k]:8.0%}")
print("  example, zero-shot unconstrained:", repr(runs["zero-shot, unconstrained"][0]))
print("  example, two-shot unconstrained: ", repr(runs["two-shot, unconstrained"][0]))
print("  example, zero-shot constrained:  ", repr(runs["zero-shot, constrained"][0]))
print("  example, two-shot constrained:   ", repr(runs["two-shot, constrained"][0]))

<details><summary><b>提示 1 / Hint 1</b></summary>

`text.find('{')` 和 `text.find('}')` 给出第一个左括号和第一个右括号的位置。

`text.find('{')` and `text.find('}')` give the positions of the first opening and the first closing brace.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

Python 中 `True` 是 `int` 的实例，所以判断整数要用 `type(value) is int`。

In Python `True` is an instance of `int`, so integers are tested with `type(value) is int`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def extract_json(text):
    start, end = text.find("{"), text.find("}")
    if start == -1 or end < start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def conforms(obj, schema):
    props = schema["properties"]
    if not isinstance(obj, dict) or set(obj) != set(props):
        return False
    for name, spec in props.items():
        value = obj[name]
        if "enum" in spec:
            ok = value in spec["enum"]
        elif spec["type"] == "boolean":
            ok = type(value) is bool
        elif spec["type"] == "integer":
            ok = type(value) is int                              # bool is a subclass of int in Python, so no isinstance
        else:
            ok = type(value) is str
        if not ok:
            return False
    return True


def format_error_rate(outputs, schema):
    bad = sum(1 for text in outputs if not conforms(extract_json(text), schema))
    return bad / len(outputs)


def field_accuracy(outputs, field, labels):
    hits = 0
    for text, label in zip(outputs, labels):
        obj = extract_json(text)
        hits += isinstance(obj, dict) and obj.get(field) == label
    return hits / len(outputs)
```

</details>